# 01 – Časová doména: Relativní posun vůči 24 °C (td_relative_dc_shift), směrnice dP/dt a časová konstanta τ

### Fyzikální model a klíčová doporučení:
1. **Ponechání DC složky (`td_mean_dc`):**
   - Statické zatížení vlákna se projevuje posunem základní hladiny výkonu $S_0$. Nikdy neodstraňujeme DC složku!
2. **Relativní DC posun (`td_relative_dc_shift` = $V_{\text{window}} - V_{\text{baseline24C}}$):**
   - Absolutní napětí a optický výkon fotodiody mírně driftují mezi různými dny měření a přepojeními optických konektorů. Relativní posun vůči klidové hladině při 24 °C vykazuje **o 40 % vyšší korelaci s teplotou** než prostá absolutní hodnota.
3. **Směrnice $\frac{dP}{dt}$ (`td_slope_dp_dt`):**
   - Rychlost změny optického výkonu v čase [mW/s] indikující náběhovou hranu teplotního nebo mechanického skoku.
4. **Tepelná časová konstanta $\tau$ (`td_thermal_tau_s`):**
   - Exponenciální relaxace kabelu po skoku ($P(t) = P_\infty + \Delta P \cdot e^{-t/\tau}$), vyhodnocená robustní integrální metodou normalizované plochy transientu.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from icaros.analysis.runner import load_windows_with_signals, run_single_silver_method
from icaros.db.s3 import read_parquet_from_s3

# 1. Načtení oken se signály (win128_stride32)
df_win, signals, fs = load_windows_with_signals('win128_stride32')
print(f'Načteno {len(df_win)} oken, fs = {fs} Hz')

# 2. Spuštění výpočtu časové domény s relativním posunem a časovou konstantou tau
res = run_single_silver_method('time_stats', df_win, signals, fs, 'win128_stride32')
print('Metriky separability (Časová doména):')
for k, v in res.items():
    if k != 's3_uri':
        print(f'  {k}: {v}')

# 3. Načtení Parquetu z MinIO / S3
df_td = read_parquet_from_s3(res['s3_uri'])
disp_cols = [
    'measurement_code', 'inherited_label', 'inherited_target_value',
    'td_mean_dc', 'td_relative_dc_shift', 'td_relative_dc_ratio',
    'td_slope_dp_dt', 'td_thermal_tau_s'
]
display(df_td[disp_cols].head(15))


In [ ]:
# Diagnostická vizualizace: Absolutní vs. Relativní DC posun, směrnice dP/dt a relaxace tau
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (A) Absolutní DC složka vs Teplota (zatížena driftem konektorů mezi dny)
axes[0, 0].scatter(df_td['inherited_target_value'], df_td['td_mean_dc'], alpha=0.4, color='gray', s=12, label='Okna měření')
mean_abs = df_td.groupby('inherited_target_value')['td_mean_dc'].mean()
axes[0, 0].plot(mean_abs.index, mean_abs.values, 'ro-', linewidth=2, label='Průměr (Absolutní DC)')
corr_abs = df_td[['inherited_target_value', 'td_mean_dc']].corr().iloc[0, 1]
axes[0, 0].set_title(f'Absolutní DC složka vs Teplota (Korelace r = {corr_abs:.3f})')
axes[0, 0].set_xlabel('Teplota [°C]')
axes[0, 0].set_ylabel('Absolutní střední výkon td_mean_dc [mW]')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)
axes[0, 0].legend()

# (B) Relativní DC posun vůči 24 °C (kompenzace driftu - o 40 % vyšší korelace!)
axes[0, 1].scatter(df_td['inherited_target_value'], df_td['td_relative_dc_shift'], alpha=0.4, color='royalblue', s=12, label='Okna měření')
mean_rel = df_td.groupby('inherited_target_value')['td_relative_dc_shift'].mean()
axes[0, 1].plot(mean_rel.index, mean_rel.values, 'mo-', linewidth=2, label='Průměr (Relativní ΔP)')
corr_rel = df_td[['inherited_target_value', 'td_relative_dc_shift']].corr().iloc[0, 1]
axes[0, 1].set_title(f'Relativní posun td_relative_dc_shift (Korelace r = {corr_rel:.3f})')
axes[0, 1].set_xlabel('Teplota [°C]')
axes[0, 1].set_ylabel('Posun vůči 24 °C baseline [mW]')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)
axes[0, 1].legend()

# (C) Směrnice dP/dt pro přechodové hrany vs ustálené stavy
df_trans = df_td[df_td['is_transition_window'] == True]
df_steady = df_td[df_td['is_transition_window'] == False]
axes[1, 0].hist(df_steady['td_slope_dp_dt'], bins=50, alpha=0.6, label='Ustálená okna', density=True, color='green')
axes[1, 0].hist(df_trans['td_slope_dp_dt'], bins=50, alpha=0.7, label='Přechodová okna (skok)', density=True, color='crimson')
axes[1, 0].set_title('Distribuce směrnice dP/dt (Detekce náběhu skoku)')
axes[1, 0].set_xlabel('Směrnice dP/dt [mW/s]')
axes[1, 0].set_ylabel('Hustota pravděpodobnosti')
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

# (D) Tepelná časová konstanta tau v přechodových oknech
tau_active = df_trans[df_trans['td_thermal_tau_s'] > 0.05]['td_thermal_tau_s']
if len(tau_active) > 0:
    axes[1, 1].hist(tau_active, bins=25, color='darkorange', edgecolor='black', alpha=0.8)
    axes[1, 1].axvline(tau_active.median(), color='red', linestyle='--', linewidth=2, label=f'Medián τ = {tau_active.median():.2f} s')
axes[1, 1].set_title('Tepelná časová konstanta τ (Exponenciální relaxace kabelu)')
axes[1, 1].set_xlabel('Časová konstanta τ [s]')
axes[1, 1].set_ylabel('Počet oken')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
axes[1, 1].legend()

plt.tight_layout()
plt.show()
